In [0]:
from pyspark.sql import functions as F

# Read the clean Silver journal entry table
df = spark.read.table("workspace.silver.journal_entries")

# Test 1: Duplicate Invoices
duplicate_invoices = (
    df.groupBy("LIFNR", "BELNR", "DMBTR")
    .agg(F.count("*").alias("frequency"))
    .filter("frequency > 1")
)
display(duplicate_invoices)

# Test 2: Round Amounts
round_amounts = df.filter((F.col("DMBTR") >= 1000) & (F.col("DMBTR") % 1000 == 0))
display(round_amounts)

# Test 3: Weekend Postings (Entries posted on Saturdays or Sundays)
weekend_postings = df.filter(F.dayofweek("BUDAT").isin([1, 7]))
display(weekend_postings)

# Test 4: Below-Threshold Entries
below_threshold = df.filter((F.col("DMBTR") > 0) & (F.col("DMBTR") < 1000))

# Test 5: Unbalanced Entries (Documents where total debits != total credits)
doc_balance = df.groupBy("BELNR").agg(
    F.sum(F.when(F.col("SHKZG") == "S", F.col("DMBTR")).otherwise(0)).alias("total_debit"),
    F.sum(F.when(F.col("SHKZG") == "H", F.col("DMBTR")).otherwise(0)).alias("total_credit")
)
unbalanced_entries = doc_balance.filter(F.col("total_debit") != F.col("total_credit"))
display(unbalanced_entries)

# Test 6: Benford's Law Analysis
benford_df = df.withColumn("first_digit", F.substring(F.regexp_replace(F.col("DMBTR").cast("string"), "[^0-9]", ""), 1, 1).cast("int"))
benford_dist = benford_df.filter((F.col("first_digit") >= 1) & (F.col("first_digit") <= 9)) \
    .groupBy("first_digit").agg(F.count("*").alias("actual_count"))
display(benford_dist)

LIFNR,BELNR,DMBTR,frequency
100058,5100000000,2198.51,2
100146,5100000001,1745.44,2
100089,5100000002,4511.73,2
100054,5100000003,637.35,2
100116,5100000004,15623.57,2
100125,5100000005,7148.01,2
100059,5100000006,11265.9,2
100144,5100000007,5809.75,2
100009,5100000008,1578.96,2
100042,5100000009,19244.76,2


LIFNR,BELNR,BUZEI,HKONT,SHKZG,DMBTR,WAERS,BUKRS,GJAHR,BLART,BUDAT,USNAM,XBLNR,BLDAT,NAME1,LAND1,ORT01,SAKNR,TXT20,KTOKS,SIGNED_AMT,POSTING_MONTH,IS_WEEKEND,_processed_at
100028,5100000415,1,500200,S,50000.0,INR,1000,2025,KR,2025-10-14,USER08,INV-700415,2025-10-14,Delta Systems 28,IN,Kolkata,500200,Repairs,EXP,50000.0,2025-10,false,2026-09-29T15:50:58.468Z
100107,5100001544,1,400100,S,50000.0,INR,1000,2025,KR,2026-03-31,USER06,INV-701544,2026-03-28,Omega Logistics 107,US,Kolkata,400100,Raw Materials,EXP,50000.0,2026-03,false,2026-09-29T15:50:58.468Z
100118,5100002401,1,400400,S,50000.0,INR,1000,2025,KR,2025-06-27,USER02,INV-702401,2025-06-25,Sigma Systems 118,IN,Mumbai,400400,Consulting Fees,EXP,50000.0,2025-06,false,2026-09-29T15:50:58.468Z
100085,5100003169,1,400100,S,50000.0,INR,1000,2025,SA,2025-06-11,USER02,INV-703169,2025-06-09,Nova Enterprises 85,IN,UNKNOWN,400100,Raw Materials,EXP,50000.0,2025-06,false,2026-09-29T15:50:58.468Z
100103,5100003253,1,400200,S,50000.0,INR,1000,2025,KR,2025-11-23,USER12,INV-703253,2025-11-18,Omega Solutions 103,IN,Bengaluru,400200,Office Expenses,EXP,50000.0,2025-11,true,2026-09-29T15:50:58.468Z
100092,5100003729,1,400200,S,50000.0,INR,1000,2025,KR,2026-02-18,USER20,INV-703729,2026-02-17,Vertex Solutions 92,IN,Hyderabad,400200,Office Expenses,EXP,50000.0,2026-02,false,2026-09-29T15:50:58.468Z
100001,5100004244,1,400100,S,50000.0,INR,1000,2025,SA,2025-05-28,USER18,INV-704244,2025-05-27,Bharat Industries 1,IN,Kochi,400100,Raw Materials,EXP,50000.0,2025-05,false,2026-09-29T15:50:58.468Z
100021,5100004570,1,400400,S,50000.0,INR,1000,2025,KR,2025-11-06,USER13,INV-704570,2025-11-02,Bharat Services 21,IN,Bengaluru,400400,Consulting Fees,EXP,50000.0,2025-11,false,2026-09-29T15:50:58.468Z
100017,5100004895,1,400300,S,50000.0,INR,1000,2025,KR,2026-01-30,USER10,INV-704895,2026-01-29,Vertex Traders 17,SG,Ahmedabad,400300,Travel,EXP,50000.0,2026-01,false,2026-09-29T15:50:58.468Z
100110,5100005642,1,400100,S,50000.0,INR,1000,2025,SA,2025-09-18,USER19,INV-705642,2025-09-16,Apex Traders 110,SG,Pune,400100,Raw Materials,EXP,50000.0,2025-09,false,2026-09-29T15:50:58.468Z


LIFNR,BELNR,BUZEI,HKONT,SHKZG,DMBTR,WAERS,BUKRS,GJAHR,BLART,BUDAT,USNAM,XBLNR,BLDAT,NAME1,LAND1,ORT01,SAKNR,TXT20,KTOKS,SIGNED_AMT,POSTING_MONTH,IS_WEEKEND,_processed_at
100146,5100000001,2,210000,H,1745.44,INR,1000,2025,KR,2025-06-08,USER01,INV-700001,2025-06-07,Titan Traders 146,IN,Kolkata,210000,Accounts Payable,LIAB,-1745.44,2025-06,true,2026-09-29T15:50:58.468Z
100146,5100000001,1,400100,S,1745.44,INR,1000,2025,KR,2025-06-08,USER01,INV-700001,2025-06-07,Titan Traders 146,IN,Kolkata,400100,Raw Materials,EXP,1745.44,2025-06,true,2026-09-29T15:50:58.468Z
100059,5100000006,1,400400,S,11265.9,INR,1000,2025,KR,2025-08-10,USER07,INV-700006,2025-08-10,Prime Systems 59,IN,Pune,400400,Consulting Fees,EXP,11265.9,2025-08,true,2026-09-29T15:50:58.468Z
100059,5100000006,2,210000,H,11265.9,INR,1000,2025,KR,2025-08-10,USER07,INV-700006,2025-08-10,Prime Systems 59,IN,Pune,210000,Accounts Payable,LIAB,-11265.9,2025-08,true,2026-09-29T15:50:58.468Z
100009,5100000008,2,210000,H,1578.96,INR,1000,2025,KR,2025-12-14,USER18,INV-700008,2025-12-09,Bharat Services 9,DE,Delhi,210000,Accounts Payable,LIAB,-1578.96,2025-12,true,2026-09-29T15:50:58.468Z
100009,5100000008,1,400300,S,1578.96,INR,1000,2025,KR,2025-12-14,USER18,INV-700008,2025-12-09,Bharat Services 9,DE,Delhi,400300,Travel,EXP,1578.96,2025-12,true,2026-09-29T15:50:58.468Z
100044,5100000016,2,210000,H,2345.41,INR,1000,2025,SA,2025-08-10,USER20,INV-700016,2025-08-10,Prime Solutions 44,IN,Kochi,210000,Accounts Payable,LIAB,-2345.41,2025-08,true,2026-09-29T15:50:58.468Z
100044,5100000016,1,500100,S,2345.41,INR,1000,2025,SA,2025-08-10,USER20,INV-700016,2025-08-10,Prime Solutions 44,IN,Kochi,500100,IT Services,EXP,2345.41,2025-08,true,2026-09-29T15:50:58.468Z
100048,5100000017,2,210000,H,4836.21,INR,1000,2025,SA,2025-07-12,USER17,INV-700017,2025-07-08,Apex Services 48,IN,Mumbai,210000,Accounts Payable,LIAB,-4836.21,2025-07,true,2026-09-29T15:50:58.468Z
100048,5100000017,1,400300,S,4836.21,INR,1000,2025,SA,2025-07-12,USER17,INV-700017,2025-07-08,Apex Services 48,IN,Mumbai,400300,Travel,EXP,4836.21,2025-07,true,2026-09-29T15:50:58.468Z


BELNR,total_debit,total_credit
5100000074,20101.52,15223.52
5100000310,49756.96,1095.27
5100000415,50000.0,28618.15
5100000527,49546.94,3225.64
5100000770,1753.46,-3089.54
5100000836,48589.46,529.22
5100001124,2574.82,-1360.1799999999998
5100001487,46782.46,45649.46
5100001497,48236.0,5777.01
5100001544,50000.0,1328.33


first_digit,actual_count
2,6820
1,11982
4,4043
6,2712
7,2378
5,3196
3,5109
8,1905
9,1855
